# Weather History — Sanity Check Notebook

Use this notebook to quickly sanity-check your Bronze weather history parquet output.

**What it does**
- Loads a Parquet *file or directory* (partitioned Parquet supported)
- Prints schema and basic row counts
- Detects a datetime column and computes date coverage
- Aggregates row counts by day and by month (with plots)
- Summarizes nulls and basic stats for numeric columns
- Shows top stations and per-station counts

> Tip: If your dataset is a directory of partitioned Parquet files, just set `PARQUET_PATH` to the directory.

In [7]:
# ==== Parameters (edit here) ====
# Path to your weather history parquet (file or directory)
PARQUET_PATH = "../../datamart/bronze/weather_history"  # change if needed

# Optional: limit reads for speed during exploration (None = no limit)
ROW_LIMIT = None  # e.g., 2_000_000

# If your datetime column name is known, set it here to override auto-detection
FORCE_DATETIME_COL = None  # e.g., "valid_date" or "date"


In [8]:
import os
import math
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

print(pd.__version__, 'pandas')
print(np.__version__, 'numpy')
print(pa.__version__, 'pyarrow')


2.0.3 pandas
1.24.4 numpy
13.0.0 pyarrow


In [9]:
# ---- Load Parquet (file or directory) ----
assert os.path.exists(PARQUET_PATH), f'Path not found: {PARQUET_PATH}'

# Use Arrow dataset for both directory and single-file cases
dataset = ds.dataset(PARQUET_PATH, format='parquet')

# Get schema and row count fast
schema = dataset.schema
try:
    # For partitioned dirs, this can be faster than full scan depending on metadata
    fragments = list(dataset.get_fragments())
    num_files = len(fragments)
except Exception:
    fragments, num_files = [], None

print("Schema:")
print(schema)
print("\nNum files:", num_files)

# Read into pandas (optionally with a limit)
table = ds.Scanner.from_dataset(dataset).to_table()
if ROW_LIMIT is not None and table.num_rows > ROW_LIMIT:
    table = table.slice(0, ROW_LIMIT)
df = table.to_pandas(types_mapper=pd.ArrowDtype)
print("Loaded rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

Schema:
STATION: string
DATE: string
SOURCE: string
LATITUDE: string
LONGITUDE: string
ELEVATION: string
NAME: string
REPORT_TYPE: string
CALL_SIGN: string
QUALITY_CONTROL: string
WND: string
CIG: string
VIS: string
TMP: string
DEW: string
SLP: string
AA1: string
AA2: string
AA3: string
AB1: string
AD1: string
AE1: string
AH1: string
AH2: string
AH3: string
AH4: string
AH5: string
AH6: string
AI1: string
AI2: string
AI3: string
AI4: string
AI5: string
AI6: string
AJ1: string
AK1: string
AL1: string
AM1: string
AN1: string
AT1: string
AT2: string
AT3: string
AT4: string
AT5: string
AU1: string
AU2: string
AU3: string
AW1: string
AW2: string
AW3: string
AW4: string
AX1: string
AX2: string
AX3: string
ED1: string
GA1: string
GA2: string
GA3: string
GA4: string
GD1: string
GD2: string
GD3: string
GD4: string
GE1: string
GF1: string
KA1: string
KA2: string
KB1: string
KB2: string
KB3: string
KC1: string
KC2: string
KD1: string
KD2: string
KE1: string
KG1: string
KG2: string
MA1: string
MD1:

,STATION,DATE,SOURCE,LATITUDE,LONGITUDE,ELEVATION,NAME,REPORT_TYPE,CALL_SIGN,QUALITY_CONTROL,...,OE2,OE3,RH1,RH2,RH3,REM,EQD,obs_ts,snapshot_date,station_id
0,72502014734,2023-01-01T00:51:00,7,40.68275,-74.16927,1.9,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",FM-15,KEWR,V030,...,<NA>,<NA>,<NA>,<NA>,<NA>,MET15012/31/22 19:51:03 METAR KEWR 010051Z 220...,<NA>,2023-01-01 00:51:00,2023-01-01,72502014734
1,72502014734,2023-01-01T01:51:00,7,40.68275,-74.16927,1.9,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",FM-15,KEWR,V030,...,<NA>,<NA>,<NA>,<NA>,<NA>,MET14012/31/22 20:51:03 METAR KEWR 010151Z 180...,<NA>,2023-01-01 01:51:00,2023-01-01,72502014734
2,72502014734,2023-01-01T02:49:00,6,40.68275,-74.16927,1.9,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",FM-16,KEWR,V030,...,<NA>,<NA>,<NA>,<NA>,<NA>,MET13512/31/22 21:49:03 SPECI KEWR 010249Z 230...,<NA>,2023-01-01 02:49:00,2023-01-01,72502014734


In [18]:
# ---- Try to detect a datetime column ----
def try_find_datetime_col(columns, df):
    if FORCE_DATETIME_COL and FORCE_DATETIME_COL in df.columns:
        return FORCE_DATETIME_COL
    candidates = [c for c in df.columns if any(k in c.lower() for k in ['DATE'])]
    # prefer columns already datetime64
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.datetime64):
            return c
    # then try candidates by parsing a sample
    for c in candidates:
        try:
            pd.to_datetime(df[c].iloc[:100], errors='raise')
            return c
        except Exception:
            continue
    # fallback: None
    return None

dt_col = try_find_datetime_col(df.columns, df)
print("Detected datetime column:", dt_col)
if dt_col is not None:
    # coerce to datetime
    df[dt_col] = pd.to_datetime(df[dt_col], errors='coerce', utc=True)
    print("Date range:", df[dt_col].min(), "->", df[dt_col].max())
else:
    print("No datetime column detected. Set FORCE_DATETIME_COL to override.")


TypeError: Cannot interpret 'string[pyarrow]' as a data type

In [13]:
# ---- Basic nulls and dtypes ----
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'null_count': df.isna().sum(),
    'non_null': df.notna().sum()
}).sort_values('null_count', ascending=False)
summary.head(20)

,dtype,null_count,non_null
AH3,string[pyarrow],78708,55
AH1,string[pyarrow],78708,55
AH2,string[pyarrow],78708,55
AH6,string[pyarrow],78708,55
AH5,string[pyarrow],78708,55
AH4,string[pyarrow],78708,55
AI6,string[pyarrow],78707,56
AI5,string[pyarrow],78707,56
AI4,string[pyarrow],78707,56
AI3,string[pyarrow],78707,56


In [14]:
# ---- Numeric stats (quick) ----
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc = df[num_cols].describe().T if num_cols else pd.DataFrame()
desc.head(20)

""


In [15]:
# ---- Station/airport popularity (best-guess columns) ----
station_like = None
for k in ['station','icao','wban','usaf','sid','station_id','airport']:
    cands = [c for c in df.columns if k in c.lower()]
    if cands:
        station_like = cands[0]
        break

if station_like:
    top_stations = df[station_like].value_counts().head(20)
    print("Station-like column:", station_like)
    display(top_stations)
else:
    print("No obvious station/airport column found.")

Station-like column: STATION


STATION
72503014732    27061
74486094789    26471
72502014734    25231
Name: count, dtype: int64[pyarrow]

In [16]:
# ---- Aggregations by day/month with plots ----
if dt_col is not None:
    df['_date'] = df[dt_col].dt.tz_convert(None) if str(df[dt_col].dtype).startswith('datetime64[ns,') else df[dt_col]
    df['_day'] = df['_date'].dt.date
    df['_month'] = df['_date'].dt.to_period('M').dt.to_timestamp()

    by_day = df.groupby('_day', as_index=False).size()
    by_month = df.groupby('_month', as_index=False).size()

    # Show head
    display(by_day.head()), display(by_month.head())

    # Plot daily counts
    plt.figure(figsize=(10,3))
    plt.plot(by_day['_day'], by_day['size'])
    plt.title('Row count by day')
    plt.xlabel('Day')
    plt.ylabel('Rows')
    plt.tight_layout()
    plt.show()

    # Plot monthly counts
    plt.figure(figsize=(8,3))
    plt.plot(by_month['_month'], by_month['size'])
    plt.title('Row count by month')
    plt.xlabel('Month')
    plt.ylabel('Rows')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping day/month plots: no datetime column detected.")

NameError: name 'dt_col' is not defined

In [17]:
# ---- Per-station per-day pivot sample (top N stations) ----
if dt_col is not None and station_like:
    # pick top 10 stations by volume
    topN = df[station_like].value_counts().head(10).index
    small = df[df[station_like].isin(topN)].copy()
    small['_day'] = small[dt_col].dt.date
    piv = small.pivot_table(index='_day', columns=station_like, values=dt_col, aggfunc='count').fillna(0).astype(int)
    piv.head()
else:
    print("Skipping station/day pivot: need both datetime and station columns.")

NameError: name 'dt_col' is not defined

In [ ]:
# ---- Optional: write small profile CSV next to notebook output ----
out_csv = "/mnt/data/weather_hist_profile.csv"
cols = ['dtype','null_count','non_null']
summary_out = summary.reset_index().rename(columns={'index':'column'})
summary_out.to_csv(out_csv, index=False)
print("Saved summary to:", out_csv)